# Example: a partitioner from domain knowledge

k-means assumes clusters of similar size and spread. On a sample with one wide mode and two narrow modes close to
each other it cuts the wide mode in two and merges the narrow ones, and EM from such a start stays in a local optimum.
If the expert knows where the modes are separated, the partition does not need clustering at all: a partitioner of a
few lines splits the sample at the known thresholds.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

from pysatl_mpest.initializers import ClusterInitializer, IdentityMatcher, KMeansPartitioner, Partitioner

## Data

In [ ]:
truth = MixtureModel(
    [normal(mu=0, sigma=3), normal(mu=15, sigma=0.3), normal(mu=17, sigma=0.3)], weights=[0.7, 0.15, 0.15]
)
X = truth.generate(size=600, random_state=1)
families = [normal] * 3

## The partitioner

The contract of `Partitioner.partition(X, k, rng)`: a matrix of shape `(n, k)` with exactly one 1 in a row, and all
randomness from `rng` — here there is none. The columns go from left to right, so the families are listed from left
to right too, and `IdentityMatcher` gives column `j` to family `j`. A wrong number of clusters is an error of the
configuration, so it is a `ValueError`, not an `InitializationError`.

In [ ]:
class ThresholdPartitioner(Partitioner):
    def __init__(self, thresholds: list[float]) -> None:
        self.thresholds = np.asarray(thresholds, dtype=np.float64)

    def partition(self, X: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
        if k != len(self.thresholds) + 1:
            raise ValueError(f"{len(self.thresholds)} thresholds make {len(self.thresholds) + 1} clusters, not {k}.")
        return np.eye(k)[np.searchsorted(self.thresholds, X)]


expert = ClusterInitializer(ThresholdPartitioner([12.0, 16.0]), matcher=IdentityMatcher())
kmeans = ClusterInitializer(KMeansPartitioner())
starts = {"thresholds": expert.initialize(X, families, random_state=0), "k-means": kmeans.initialize(X, families, random_state=0)}
fits = {name: em(100).fit(X, start) for name, start in starts.items()}
for name in starts:
    print(f"{name}: start {starts[name].loglikelihood(X):.1f}, after EM {fits[name].loglikelihood(X):.1f}")
    show(fits[name])
print("true mixture:", round(truth.loglikelihood(X), 1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.5), sharey=True)
for ax, name in zip(axes, starts):
    plot_mixtures(ax, X, {"start": starts[name], "after EM": fits[name]}, bins=100)
    ax.set_title(name)
plt.show()

try:
    expert.initialize(X, [normal] * 2, random_state=0)
except ValueError as error:
    print("configuration error:", error)

assert fits["thresholds"].loglikelihood(X) > fits["k-means"].loglikelihood(X) + 50

## Takeaway

A partitioner is the place for domain knowledge: anything that returns one cluster per point works with the
matchers, the checks and the multistart of the library. When the partitioner already knows which cluster belongs to
which family, `IdentityMatcher` keeps that order.